# Track V — V2b-4: v2 search inputs, rehearsal and search (notebook 08)

This notebook produces every input of the model v2 search on **one fixed commit and one Colab T4 runtime 2026.10** (D58, D64), then runs the T4 **rehearsal** and the D49 **search** with `python -m src.tuning.v2_search` ([docs/v2_training.md](../docs/v2_training.md)). Decisions: D9 (train and val-A only), D40 (W&B), D42, D49, D50/D56, D58, D61, D64.

- **Inputs, one cell each, "build once if absent, otherwise verify":** the raw train cache (V2b-3), the FoR archive, the Stage 8 train and val-A log-Mel caches, the MUSAN and RIRS_NOISES archives, the selection set S and the training-half resource cache (both V1c, built by Malak's reviewed builders, called unchanged), then the small V1c evidence files. Every Drive output is written once (never overwritten) and re-hashed after the copy.
- **Fixed commit:** the provenance of every v2 run includes the code commit, so the whole search (5 runs, worst case 5–8 h over several Colab sessions) runs from the same `COMMIT`. HEAD and a clean worktree are asserted before and after every long cell.
- **Re-running after a disconnect:** open a new session on the same runtime, keep `COMMIT` and run all cells from the top. The input cells only verify what is already on Drive, the rehearsal cell shows the rehearsal already done for `COMMIT` instead of running it again, and the search cell resumes (finished runs are verified and reused, a stopped run resumes at its next epoch; PR V2b-R).

Only FoR train and val-A (through S) are read. The notebook never names any other split. Audio, caches, checkpoints and archives stay on Drive and under `/content`, never in the repository (AGENTS.md rule 7).

## 1. Parameters

Select the Colab runtime **T4 GPU** and, under "Version d'exécution / Runtime version", **2026.10** before running (D58 (1)). Replace `COMMIT` with the full 40-character commit of the merged `main` that holds this notebook. That is the only value to fill. `SEARCH_VERSION` (`model-v2`, the model version the SRS uses and F5 takes from the search reports) names the search folder and the W&B runs and stays the same for the whole search; `NUM_WORKERS` is the DataLoader worker count (the Colab T4 has 2 vCPUs).

`CORPUS_DIR` is Malak's shared V1c corpus folder. It is reached through a **My Drive shortcut named exactly `voice-spoofing-detection-v1b-checkpoint`** (right-click the shared folder, "Add shortcut to Drive", My Drive). It must hold `musan.tar.gz` and `rirs_noises.zip`. It is only read, and only when S or the resource cache still has to be built.

In [ ]:
REPO_URL = 'https://github.com/kahlanzzi/voice-spoofing-detection.git'
COMMIT = 'REPLACE_WITH_MERGED_MAIN_COMMIT'  # full 40-character SHA of the merged main
SEARCH_VERSION = 'model-v2'  # the F5 model version (SRS: model-v2); recorded in the search reports
NUM_WORKERS = 2
COLAB_RUNTIME_VERSION = '2026.10'  # D58 (1); passed to the V1c S builder, which records it
DRIVE_ROOT = '/content/drive/MyDrive/voice-spoofing-detection'
CORPUS_DIR = '/content/drive/MyDrive/voice-spoofing-detection-v1b-checkpoint'
assert COMMIT and COMMIT != 'REPLACE_WITH_MERGED_MAIN_COMMIT', 'fill COMMIT with the merged main commit'
assert len(COMMIT) == 40, f'COMMIT must be the full 40-character SHA, got {COMMIT!r}'
assert SEARCH_VERSION != 'rehearsal', 'the rehearsal folder name is reserved'

## 2. Mount Drive; the Track V Drive layout

All paths are constants under `DRIVE_ROOT` (the "Drive layout (Track V)" table in `docs/v2_training.md` lists every folder with its writer and readers). This notebook writes only the folders marked *written* below; the v1 folders (`checkpoints/cnn/`, `processed/metadata/stage8/`, `stage10/`, `stage11/`, …) are never written.

| Constant | Drive path | Here |
|---|---|---|
| `RAW_CACHE_DRIVE` | `processed/v2/raw_train_cache/` (`train_raw.npz`, `train_raw.json`, V2b-3) | read |
| `STAGE8_LOGMEL_DRIVE` | `processed/v2/stage8_logmel/{train,val-A}/` | written once if absent, else read |
| `SELECTION_SET_DRIVE` | `processed/v2/selection_set_s/` | written once if absent, else read |
| `RESOURCES_DRIVE` | `processed/v2/aug_resources/` | written once if absent, else read |
| `V1C_EVIDENCE_DRIVE` | `processed/v2/v1c_evidence/` | written once |
| `SEARCH_OUT` | `checkpoints/v2_search/<SEARCH_VERSION>/` (`runs/` and the three reports) | written by the search CLI |
| `REHEARSAL_OUT` | `checkpoints/v2_search/rehearsal/<UTC stamp>/` | written by the rehearsal CLI |
| FoR archive | `raw/for_2sec_ff8c82c7.tar` and its revision marker | read |

Local copies go to `/content/v2_inputs/` (outside the clone). The search and rehearsal write straight to Drive, so a disconnect keeps every finished epoch.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path(DRIVE_ROOT)
CORPUS = Path(CORPUS_DIR)
V2 = DRIVE / 'processed' / 'v2'
RAW_CACHE_DRIVE = V2 / 'raw_train_cache'
STAGE8_LOGMEL_DRIVE = V2 / 'stage8_logmel'
SELECTION_SET_DRIVE = V2 / 'selection_set_s'
RESOURCES_DRIVE = V2 / 'aug_resources'
V1C_EVIDENCE_DRIVE = V2 / 'v1c_evidence'
SEARCH_ROOT = DRIVE / 'checkpoints' / 'v2_search'
SEARCH_OUT = SEARCH_ROOT / SEARCH_VERSION
LOCAL = Path('/content/v2_inputs')
LOCAL.mkdir(parents=True, exist_ok=True)


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def file_hashes(folder):
    return {p.relative_to(folder).as_posix(): sha256(p) for p in sorted(Path(folder).rglob('*')) if p.is_file()}


def copy_exclusive(source, dest):
    # Copy a folder to a destination that must not exist yet, then re-hash every file of the copy.
    if dest.exists():
        raise FileExistsError(f'{dest} already exists; it is never overwritten')
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, dest)
    expected, actual = file_hashes(source), file_hashes(dest)
    assert actual == expected, f'{dest} differs from {source} after the copy'
    return actual


def local_copy(drive_dir, name):
    # The Drive folder copied once per session to /content/v2_inputs/<name>; the loaders verify its content.
    target = LOCAL / name
    if not target.exists():
        shutil.copytree(drive_dir, target)
    return target


assert (DRIVE / 'raw').is_dir(), f'{DRIVE / "raw"} not found; check DRIVE_ROOT'
assert RAW_CACHE_DRIVE.is_dir(), f'{RAW_CACHE_DRIVE} not found (V2b-3 raw train cache)'
print('Drive mounted; search folder', SEARCH_OUT, 'exists' if SEARCH_OUT.exists() else 'is new')

## 3. Check out the exact commit

As notebooks 04 and 06: the repository is cloned into `/content` and `COMMIT` is checked out detached. `assert_clean` requires HEAD equal to `COMMIT`, a clean `git status` and no ignored file other than Python bytecode caches (`.gitignore` hides `wandb/`). It runs before and after every long cell. Every output is written outside the clone, so the CLIs, which refuse a dirty worktree, see a clean tree.

In [ ]:
REPO = Path('/content/voice-spoofing-detection')
assert len(COMMIT) == 40, f'COMMIT must be the full 40-character SHA, got {COMMIT!r}'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT], cwd=REPO, check=True)


def git(*args):
    return subprocess.run(['git', *args], cwd=REPO, check=True, capture_output=True, text=True).stdout


def assert_clean():
    head = git('rev-parse', 'HEAD').strip()
    assert head == COMMIT, (head, COMMIT)
    status = git('status', '--porcelain')
    assert not status, status
    ignored = [line[3:] for line in git('status', '--porcelain', '--ignored').splitlines() if line.startswith('!! ')]
    unexpected = [path for path in ignored if not path.rstrip('/').endswith('__pycache__')]
    assert not unexpected, unexpected


def stream(command, **kwargs):
    # Like subprocess.run(check=True), but prints the child's output line by line as it runs.
    env = {**kwargs.pop('env', os.environ), 'PYTHONUNBUFFERED': '1'}
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
                          **kwargs) as process:
        for line in process.stdout:
            print(line, end='')
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)


def probe(script):
    # Run a short Python script in a fresh process inside the clone; return its JSON output.
    result = subprocess.run([sys.executable, '-c', script], cwd=REPO, check=True, capture_output=True, text=True)
    return json.loads(result.stdout)


assert_clean()
os.chdir(REPO)
MANIFEST = REPO / 'data/manifests/for2sec_frozen_manifest.csv'
PROFILE = REPO / 'configs/preprocessing.yaml'
print('HEAD', COMMIT)

## 4. Runtime gate and the D64 version guard

The runtime must be a Colab **T4** with runtime **2026.10**: Python exactly 3.13.16 and Colab's preinstalled torch exactly 2.11.0+cu130 (D58 (1)). Torch is never installed: only `requirements.txt` and `requirements-cnn.txt` are installed, and neither lists torch.

Then `src.augmentation.version_guard.require_versions()` runs **in a fresh process**, after the install: NumPy, SciPy and SoundFile must equal the `requirements.txt` pins (D64: notebook 07 ran with Colab's preinstalled SciPy 1.16.3 instead of 1.18.1, which changed the reverberation bytes). Its result is printed; any mismatch stops the notebook here. This is the same guard the V1c builders, `run_search` and the rehearsal call; the notebook has no version helper of its own.

In [ ]:
import torch

GPU_NAME = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                          check=True, capture_output=True, text=True).stdout.strip()
assert 'T4' in GPU_NAME, f'a Colab T4 runtime is required, got {GPU_NAME!r}'
assert torch.__version__ == '2.11.0+cu130', f'D58 (1): runtime 2026.10 (torch 2.11.0+cu130) required, got {torch.__version__}'
assert_clean()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', '-r', 'requirements-cnn.txt'],
               cwd=REPO, check=True)
ENV = probe('''
import json, platform, torch
from src.augmentation.version_guard import require_versions
print(json.dumps({
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'require_versions': require_versions(),
}))
''')
assert ENV['python'] == '3.13.16', f"D58 (1): Python 3.13.16 required, got {ENV['python']}"
assert ENV['torch'] == '2.11.0+cu130', f"D58 (1): torch 2.11.0+cu130 required, got {ENV['torch']}"
assert ENV['cuda_available'] and 'T4' in ENV['gpu_name'], ENV
VERSIONS = ENV['require_versions']
assert_clean()
print('require_versions:', VERSIONS)
for key, value in ENV.items():
    print(f'{key}: {value}')

## 5. Pinned FFmpeg 6.1.1-3ubuntu5

The codec family and the S builder use FFmpeg pinned in `configs/augmentation.yaml` (V1b). If the installed Debian package is not `7:6.1.1-3ubuntu5`, that exact version is installed with apt (the Colab 2026.10 image is expected to ship it already). Then `FFmpegCodecBackend.verify_pin()` checks the version line and every codec library line in a fresh process, as training does.

In [ ]:
FFMPEG_PACKAGE = '7:6.1.1-3ubuntu5'
installed = subprocess.run(['dpkg-query', '-W', '-f=${Version}', 'ffmpeg'], capture_output=True, text=True).stdout.strip()
if installed != FFMPEG_PACKAGE:
    subprocess.run(['apt-get', 'update', '-q'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-q', '--allow-downgrades', f'ffmpeg={FFMPEG_PACKAGE}'], check=True)
FFMPEG = probe('''
import json
from src.augmentation.codec import FFmpegCodecBackend
from src.augmentation.config import load_augmentation_config
config = load_augmentation_config('configs/augmentation.yaml')
codec = FFmpegCodecBackend(config.ffmpeg, config.sample_rate)
codec.verify_pin()
print(json.dumps({'version_line': codec.version_text().splitlines()[0]}))
''')
assert_clean()
print('ffmpeg package was', installed or 'absent', '->', FFMPEG['version_line'])

## 6. Input: raw train cache (V2b-3)

Copied from `processed/v2/raw_train_cache/` (verified on 17 Nov). The npz must hash to the `npz_sha256` of the committed V2b-3 evidence (`docs/audit_evidence/v2b3_raw_train_cache/train_raw.json`) and `train_raw.json` must equal that file byte for byte, the same check the search CLI makes (`check_raw_cache_pin`). The cache's own `verify` then re-checks every array and prints `n_all_zero_clips` (0: D62 not needed).

In [ ]:
RAW_EVIDENCE = json.loads((REPO / 'docs/audit_evidence/v2b3_raw_train_cache/train_raw.json').read_text(encoding='utf-8'))
LOCAL_RAW = local_copy(RAW_CACHE_DRIVE, 'raw_train_cache')
assert sorted(p.name for p in LOCAL_RAW.iterdir()) == ['train_raw.json', 'train_raw.npz'], sorted(LOCAL_RAW.iterdir())
RAW_NPZ_SHA256 = sha256(LOCAL_RAW / 'train_raw.npz')
assert RAW_NPZ_SHA256 == RAW_EVIDENCE['npz_sha256'], f"raw cache npz {RAW_NPZ_SHA256} != V2b-3 evidence {RAW_EVIDENCE['npz_sha256']}"
assert sha256(LOCAL_RAW / 'train_raw.json') == sha256(REPO / 'docs/audit_evidence/v2b3_raw_train_cache/train_raw.json')
assert_clean()
stream([sys.executable, '-m', 'src.features.raw_train_cache', 'verify', '--dir', str(LOCAL_RAW)], cwd=REPO)
assert_clean()
print('raw train cache npz', RAW_NPZ_SHA256)

## 7. Input: FoR archive

As notebooks 04 and 06: the revision marker next to the Drive backup must equal the D21 FoR revision in `configs/audit.yaml`; the archive is extracted with `filter='data'`; its SHA-256 is recorded as `for_archive_sha256_observed` (evidence only, never asserted, not a pin; D21 unchanged). It comes before the log-Mel caches and S because both builds read FoR audio; every file hash is checked against F2 before decoding.

In [ ]:
import tarfile
import yaml

EXPECTED_REVISION = yaml.safe_load((REPO / 'configs/audit.yaml').read_text(encoding='utf-8'))['for2sec']['revision']
ARCHIVE = DRIVE / 'raw' / 'for_2sec_ff8c82c7.tar'
REVISION_FILE = DRIVE / 'raw' / 'for_2sec_REVISION.txt'
assert REVISION_FILE.read_text(encoding='utf-8').strip() == EXPECTED_REVISION
DATA_ROOT = Path('/content/data')
AUDIO_ROOT = DATA_ROOT / 'for_2sec'
LOCAL_ARCHIVE = DATA_ROOT / 'for_2sec_ff8c82c7.tar'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)
if AUDIO_ROOT.exists():
    shutil.rmtree(AUDIO_ROOT)
with tarfile.open(LOCAL_ARCHIVE) as archive:
    archive.extractall(DATA_ROOT, filter='data')
FOR_ARCHIVE_SHA256_OBSERVED = sha256(LOCAL_ARCHIVE)  # recorded only (evidence, not a pin)
print('FoR archive extracted to', AUDIO_ROOT)
print('for_archive_sha256_observed:', FOR_ARCHIVE_SHA256_OBSERVED)

## 8. Input: Stage 8 log-Mel caches (train, val-A)

The search reads the Stage 8 train cache (clean-train statistics, D61 (5)) and the val-A cache (S's known answer). If `processed/v2/stage8_logmel/<split>/` is absent on Drive, it is built once with `src.features.logmel_cache` (F3 profile, F2 manifest), its `npz_sha256` must equal the Stage 8 evidence (`docs/audit_evidence/stage8_cnn/run_report.json`: `train_cache_npz_sha256`, `val_a_cache_npz_sha256`; D46 (4) and D58 (2): the T4 2026.10 build reproduces Stage 8 byte for byte), and it is copied to Drive and re-hashed. Otherwise the Drive copy is copied locally and the same hashes are checked.

In [ ]:
STAGE8 = json.loads((REPO / 'docs/audit_evidence/stage8_cnn/run_report.json').read_text(encoding='utf-8'))
STAGE8_EXPECTED = {'train': STAGE8['train_cache_npz_sha256'], 'val-A': STAGE8['val_a_cache_npz_sha256']}
LOCAL_LOGMEL = LOCAL / 'stage8_logmel'
LOGMEL_BUILT = {}
for split, expected in STAGE8_EXPECTED.items():
    drive_dir, local_dir = STAGE8_LOGMEL_DRIVE / split, LOCAL_LOGMEL / split
    LOGMEL_BUILT[split] = not drive_dir.exists()
    if LOGMEL_BUILT[split]:
        assert not local_dir.exists(), f'{local_dir} exists without a Drive copy; remove it by hand'
        assert_clean()
        stream([sys.executable, '-m', 'src.features.logmel_cache', '--root', str(AUDIO_ROOT),
                '--manifest', str(MANIFEST), '--profile', str(PROFILE), '--split', split,
                '--out', str(local_dir)], cwd=REPO)
        assert_clean()
    elif not local_dir.exists():
        shutil.copytree(drive_dir, local_dir)
    meta = json.loads((local_dir / f'{split}.json').read_text(encoding='utf-8'))
    actual = sha256(local_dir / f'{split}.npz')
    assert meta['npz_sha256'] == actual == expected, f'{split}: npz {actual}, json {meta["npz_sha256"]}, Stage 8 {expected}'
    if LOGMEL_BUILT[split]:
        copy_exclusive(local_dir, drive_dir)
    print(split, 'log-Mel', actual, '(built and saved to Drive)' if LOGMEL_BUILT[split] else '(from Drive)')

## 9. Input: MUSAN and RIRS_NOISES archives (only if S or the resource cache must be built)

Read from `CORPUS_DIR` (the shortcut to Malak's V1c folder, section 1). Each archive's SHA-256 must equal the pin in `configs/augmentation.yaml` **before** anything is extracted; a mismatch stops with both hashes. Extraction follows notebook 07: the RIRS ZIP is tested and every member path checked, then extracted; from MUSAN only `musan/noise/` is extracted, member by member, with `filter='data'`. The builders read the files listed in the committed V1b inventories and check every file hash before decoding. When both V1c folders are already on Drive, this cell does nothing.

In [ ]:
import tarfile
import zipfile
from pathlib import PurePosixPath

NEED_CORPORA = not SELECTION_SET_DRIVE.exists() or not RESOURCES_DRIVE.exists()
CORPUS_WORK = Path('/content/v2_corpora')
MUSAN_NOISE_ROOT = CORPUS_WORK / 'musan' / 'musan' / 'noise'
RIRS_ROOT = CORPUS_WORK / 'rirs_noises' / 'RIRS_NOISES'
ARCHIVE_SHA256_OBSERVED = {}
if NEED_CORPORA:
    PINS = yaml.safe_load((REPO / 'configs/augmentation.yaml').read_text(encoding='utf-8'))['corpora']
    assert shutil.disk_usage('/content').free >= 30 * 1024**3, 'at least 30 GB free are needed'
    CORPUS_WORK.mkdir(parents=True, exist_ok=False)
    for name, filename in (('noise', 'musan.tar.gz'), ('rir', 'rirs_noises.zip')):
        source = CORPUS / filename
        assert source.is_file(), f'{source} not found: add the shortcut to the V1c folder (section 1)'
        local = CORPUS_WORK / filename
        shutil.copy2(source, local)
        actual, expected = sha256(local), PINS[name]['archive_sha256']
        assert actual == expected, f'{filename}: SHA-256 {actual} differs from the configs/augmentation.yaml pin {expected}'
        ARCHIVE_SHA256_OBSERVED[name] = actual
    with zipfile.ZipFile(CORPUS_WORK / 'rirs_noises.zip') as archive:
        assert archive.testzip() is None
        for member in archive.infolist():
            path = PurePosixPath(member.filename)
            assert not path.is_absolute() and '..' not in path.parts, member.filename
        archive.extractall(CORPUS_WORK / 'rirs_noises')
    musan_root = (CORPUS_WORK / 'musan').resolve()
    with tarfile.open(CORPUS_WORK / 'musan.tar.gz', mode='r:gz') as archive:
        for member in archive:
            path = PurePosixPath(member.name)
            assert not path.is_absolute() and '..' not in path.parts, member.name
            if not member.name.startswith('musan/noise/'):
                continue
            assert (CORPUS_WORK / 'musan' / member.name).resolve().is_relative_to(musan_root)
            archive.extract(member, path=CORPUS_WORK / 'musan', filter='data')
    assert MUSAN_NOISE_ROOT.is_dir() and RIRS_ROOT.is_dir()
    print('archives verified against the pins and extracted:', ARCHIVE_SHA256_OBSERVED)
else:
    print('S and the resource cache are on Drive: the corpus archives are not needed')

## 10. Input: selection set S (V1c)

If `processed/v2/selection_set_s/` is absent on Drive, S is built **once** with Malak's reviewed builder, called unchanged: `python -m src.augmentation.selection_set_build` on the D64 recipe, the selection halves of the corpora and the Stage 8 val-A log-Mel cache as the clean reference, into a local folder; the builder verifies both forms itself. The folder is then copied to Drive and re-hashed. In **every** session S is verified with `load_selection_set` in both forms (waveform and log-Mel) against the F2 manifest and the Stage 8 val-A known answer, and its manifest SHA-256 (`selection_set_sha256`) is printed.

In [ ]:
LOCAL_S = LOCAL / 'selection_set_s'
S_BUILT = not SELECTION_SET_DRIVE.exists()
if S_BUILT:
    assert not LOCAL_S.exists(), f'{LOCAL_S} exists without a Drive copy; remove it by hand'
    assert_clean()
    stream([sys.executable, '-m', 'src.augmentation.selection_set_build',
            '--for-root', str(AUDIO_ROOT),
            '--f2-manifest', str(MANIFEST),
            '--recipe', str(REPO / 'data/manifests/augmentation/selection_set_s.csv'),
            '--musan-noise-root', str(MUSAN_NOISE_ROOT),
            '--rirs-root', str(RIRS_ROOT),
            '--clean-reference-logmel', str(LOCAL_LOGMEL / 'val-A'),
            '--colab-runtime-version', COLAB_RUNTIME_VERSION,
            '--out', str(LOCAL_S)], cwd=REPO)
    assert_clean()
    copy_exclusive(LOCAL_S, SELECTION_SET_DRIVE)
else:
    local_copy(SELECTION_SET_DRIVE, 'selection_set_s')
S_CHECK = probe(f'''
import json
from pathlib import Path
from src.augmentation.selection_set_build import check_stage8_reference
from src.features.logmel_cache import load_logmel_cache
from src.preprocessing.profile import load_profile
from src.preprocessing.validation import load_f2_manifest
from src.training.selection_set import load_selection_set
profile = load_profile(Path('configs/preprocessing.yaml'))
f2 = load_f2_manifest(Path('data/manifests/for2sec_frozen_manifest.csv'))
reference = check_stage8_reference(load_logmel_cache(Path({str(LOCAL_LOGMEL / 'val-A')!r}), 'val-A', profile))
hashes = {{form: load_selection_set(Path({str(LOCAL_S)!r}), profile, form=form, f2_manifest=f2,
                                    clean_reference_logmel=reference).sha256
          for form in ('waveform', 'logmel')}}
print(json.dumps(hashes))
''')
assert S_CHECK['waveform'] == S_CHECK['logmel'] == sha256(LOCAL_S / 'selection_set_manifest.json'), S_CHECK
SELECTION_SET_SHA256 = S_CHECK['logmel']
assert_clean()
print('selection_set_sha256:', SELECTION_SET_SHA256, '(built and saved to Drive)' if S_BUILT else '(from Drive)')

## 11. Input: training-half resource cache (V1c)

If `processed/v2/aug_resources/` is absent on Drive, the cache is built **once** with Malak's reviewed builder, called unchanged (`python -m src.augmentation.resource_cache build`, the training halves of the committed V1b inventories), checked with its `verify` subcommand, copied to Drive and re-hashed. In **every** session it is loaded with `load_v2_resources` (`src/tuning/v2_search.py`), which also requires its inventory hashes to equal the committed `musan_inventory.csv` / `rirs_noises_inventory.csv` and its archive hashes to equal the `configs/augmentation.yaml` pins (the same check the search CLI makes).

In [ ]:
LOCAL_RESOURCES = LOCAL / 'aug_resources'
RESOURCES_BUILT = not RESOURCES_DRIVE.exists()
if RESOURCES_BUILT:
    assert not LOCAL_RESOURCES.exists(), f'{LOCAL_RESOURCES} exists without a Drive copy; remove it by hand'
    assert_clean()
    stream([sys.executable, '-m', 'src.augmentation.resource_cache', 'build',
            '--musan-noise-root', str(MUSAN_NOISE_ROOT), '--rirs-root', str(RIRS_ROOT),
            '--out', str(LOCAL_RESOURCES)], cwd=REPO)
    stream([sys.executable, '-m', 'src.augmentation.resource_cache', 'verify', '--dir', str(LOCAL_RESOURCES)], cwd=REPO)
    assert_clean()
    copy_exclusive(LOCAL_RESOURCES, RESOURCES_DRIVE)
else:
    local_copy(RESOURCES_DRIVE, 'aug_resources')
RESOURCES_META = probe(f'''
import json
from src.tuning.v2_search import load_v2_resources
_, meta = load_v2_resources({str(LOCAL_RESOURCES)!r})
print(json.dumps(meta))
''')
assert_clean()
print(json.dumps(RESOURCES_META, indent=2, sort_keys=True))

## 12. V1c evidence files

`v1c_build_report.json` records the S manifest SHA-256, the resource-cache hashes, the commit, the runtime, the `require_versions` result and which inputs this session built; it is written once to `processed/v2/v1c_evidence/` together with copies of `selection_set_manifest.json` and `training_resources.json`. In a later session the existing report must name the same commit, S and resource cache, and nothing is rewritten. The cell ends by fixing the input arguments shared by the rehearsal and the search. These small files are the permitted V1c evidence; they are committed in a separate evidence PR, not by this notebook.

In [ ]:
REPORT_PATH = V1C_EVIDENCE_DRIVE / 'v1c_build_report.json'
CURRENT = {
    'code_commit': COMMIT,
    'selection_set_sha256': SELECTION_SET_SHA256,
    'resource_cache_npz_sha256': RESOURCES_META['npz_sha256'],
    'resource_cache_json_sha256': sha256(LOCAL_RESOURCES / 'training_resources.json'),
}
if not REPORT_PATH.exists():
    V1C_EVIDENCE_DRIVE.mkdir(parents=True, exist_ok=False)
    report = {
        **CURRENT,
        'resource_cache': {k: RESOURCES_META[k] for k in ('noise_inventory_sha256', 'rir_inventory_sha256',
                                                          'noise_archive_sha256', 'rir_archive_sha256',
                                                          'noise_count', 'rir_count', 'code_commit')},
        'built_in_this_session': {'selection_set': S_BUILT, 'resource_cache': RESOURCES_BUILT,
                                  'stage8_logmel': LOGMEL_BUILT},
        'archive_sha256_observed': ARCHIVE_SHA256_OBSERVED,
        'stage8_logmel_npz_sha256': STAGE8_EXPECTED,
        'raw_train_cache_npz_sha256': RAW_NPZ_SHA256,
        'for_archive_sha256_observed': FOR_ARCHIVE_SHA256_OBSERVED,
        'runtime': {'colab_runtime_version': COLAB_RUNTIME_VERSION, 'python': ENV['python'],
                    'torch': ENV['torch'], 'gpu_name': ENV['gpu_name'],
                    'ffmpeg_version_line': FFMPEG['version_line']},
        'require_versions': VERSIONS,
        'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    }
    with open(REPORT_PATH, 'x', encoding='utf-8') as stream_out:
        stream_out.write(json.dumps(report, indent=2, sort_keys=True) + '\n')
    for source in (LOCAL_S / 'selection_set_manifest.json', LOCAL_RESOURCES / 'training_resources.json'):
        dest = V1C_EVIDENCE_DRIVE / source.name
        assert not dest.exists(), dest
        shutil.copy2(source, dest)
        assert sha256(dest) == sha256(source), dest
existing = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
for key, value in CURRENT.items():
    assert existing[key] == value, f'v1c_build_report.json {key} {existing[key]} differs from this session {value}'
for path in sorted(V1C_EVIDENCE_DRIVE.iterdir()):
    print(path.name, sha256(path))
# Every input is verified: the arguments shared by the rehearsal and the search.
INPUT_ARGS = ['--raw-cache-dir', str(LOCAL_RAW),
              '--stage8-train-logmel-dir', str(LOCAL_LOGMEL / 'train'),
              '--stage8-val-a-logmel-dir', str(LOCAL_LOGMEL / 'val-A'),
              '--selection-set-dir', str(LOCAL_S),
              '--resources-dir', str(LOCAL_RESOURCES),
              '--device', 'cuda', '--num-workers', str(NUM_WORKERS)]

## 13. Rehearsal (T4, before the search)

`python -m src.tuning.v2_search rehearse` with the same inputs, into a **new** folder `checkpoints/v2_search/rehearsal/<UTC stamp>/` (never the search folder: the CLI refuses a folder with `runs/` or a search report, and the search refuses a folder with the rehearsal marker). It calls `require_versions()` before any work and writes `rehearsal_report.json`: (a) `check_clean_transform` on 32 seeded train indices; (b) one epoch per candidate with the configured batch sizes (seconds per epoch, items/s); (c) crnn, seed 0, 2 epochs, twice: whether `best_selection_m` and the weights' SHA-256 are equal (observed only, D42); (d) the same run stopped by an injected exception during epoch 2 and resumed: whether it equals (c); (e) the cut line: 5 runs × 30 epochs × the measured epoch times against Fri 27 Nov (if they do not fit, the plan drops `aasist` first, Execution Plan v3.0 §7, a team decision). No W&B; no config is changed. Review the report before running the search. Expected time: about 10 epochs of training.

**Once per commit:** if a `checkpoints/v2_search/rehearsal/*/rehearsal_report.json` on Drive already has `inputs.code_commit` equal to `COMMIT`, the cell prints its path, SHA-256 and the `epoch_timing`, `determinism`, `forced_resume` and `cut_line` sections and skips the rehearsal; otherwise it runs it and prints the same for the new report.

In [ ]:
# A rehearsal already done on Drive for this COMMIT is shown, not repeated.
PREVIOUS = [path for path in sorted((SEARCH_ROOT / 'rehearsal').glob('*/rehearsal_report.json'))
            if json.loads(path.read_text(encoding='utf-8'))['inputs']['code_commit'] == COMMIT]
if not PREVIOUS:
    REHEARSAL_OUT = SEARCH_ROOT / 'rehearsal' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    assert not REHEARSAL_OUT.exists(), REHEARSAL_OUT
    assert_clean()
    stream([sys.executable, '-m', 'src.tuning.v2_search', 'rehearse', *INPUT_ARGS, '--out', str(REHEARSAL_OUT)], cwd=REPO)
    assert_clean()
    PREVIOUS = [REHEARSAL_OUT / 'rehearsal_report.json']
else:
    print(f'rehearsal skipped: {len(PREVIOUS)} report(s) on Drive already have code_commit {COMMIT}')
for path in PREVIOUS:
    REHEARSAL = json.loads(path.read_text(encoding='utf-8'))
    print(path.relative_to(DRIVE).as_posix(), sha256(path))
    print(json.dumps({k: REHEARSAL[k] for k in ('epoch_timing', 'determinism', 'forced_resume', 'cut_line')}, indent=2))

## 14. W&B credentials and runtime folders

As notebook 04: the API key is read from the Colab secret `WANDB_API_KEY` (Secrets panel) into the environment, where wandb reads it in the search process (D40). It is never printed. W&B's folders are placed under `/content/wandb_runtime`, outside the clone. The key is loaded after the rehearsal, which logs nothing.

In [ ]:
from google.colab import userdata

_key = userdata.get('WANDB_API_KEY')
assert _key and _key.strip(), 'the W&B Colab secret is missing or empty'
os.environ['WANDB_API_KEY'] = _key
del _key
os.environ['WANDB_DIR'] = '/content/wandb_runtime/run'
os.environ['WANDB_CACHE_DIR'] = '/content/wandb_runtime/cache'
os.environ['WANDB_CONFIG_DIR'] = '/content/wandb_runtime/config'
for name in ('WANDB_DIR', 'WANDB_CACHE_DIR', 'WANDB_CONFIG_DIR'):
    folder = Path(os.environ[name])
    assert not folder.resolve().is_relative_to(REPO.resolve()), folder
    folder.mkdir(parents=True, exist_ok=True)
print('W&B key loaded from Colab secrets (not shown); runtime folders under /content/wandb_runtime')

## 15. Search (safe to re-run after a disconnect)

`python -m src.tuning.v2_search search` into `checkpoints/v2_search/<SEARCH_VERSION>/` on Drive, W&B online. Each candidate once with seed 0, the lowest M wins (D49 tie rule), then the winner's seeds 1 and 2. After a disconnect, run the notebook again from the top with the same `COMMIT` (the rehearsal cell then only shows the existing report) and then this cell: complete runs are verified and reused, a stopped run resumes after its last saved epoch, a run stopped before its first epoch restarts, and the reports are written once. A different commit makes every saved state refuse to resume. Nothing here reads or reacts to M values.

In [ ]:
assert_clean()
stream([sys.executable, '-m', 'src.tuning.v2_search', 'search', *INPUT_ARGS, '--out', str(SEARCH_OUT),
        '--version', SEARCH_VERSION, '--wandb-mode', 'online'], cwd=REPO)
assert_clean()

## 16. Reports on Drive and their SHA-256

The search CLI writes `search_report.json`, `selection_report.json` and `run_report.json` straight into the search folder on Drive (they must sit next to `runs/` for the resume checks). This cell flushes Drive, remounts it and hashes the reports, the rehearsal report and the V1c evidence files as Drive stored them. The printout is the evidence for the search evidence PR (`docs/audit_evidence/v2_search/`) and the V1c evidence PR. Checkpoints are never committed.

In [ ]:
assert_clean()
REPORTS = ['search_report.json', 'selection_report.json', 'run_report.json']
RUN_REPORT = json.loads((SEARCH_OUT / 'run_report.json').read_text(encoding='utf-8'))
assert RUN_REPORT['code_commit'] == COMMIT, RUN_REPORT['code_commit']
assert RUN_REPORT['selection_set_sha256'] == SELECTION_SET_SHA256, RUN_REPORT['selection_set_sha256']
assert RUN_REPORT['raw_train_cache_npz_sha256'] == RAW_NPZ_SHA256, RUN_REPORT['raw_train_cache_npz_sha256']
EXPECTED = {SEARCH_OUT / name: sha256(SEARCH_OUT / name) for name in REPORTS}
EXPECTED.update({path: sha256(path) for path in sorted(V1C_EVIDENCE_DRIVE.iterdir())})
EXPECTED.update({path: sha256(path) for path in sorted((SEARCH_ROOT / 'rehearsal').glob('*/rehearsal_report.json'))})
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
for path, digest in EXPECTED.items():
    assert sha256(path) == digest, f'{path} changed after the Drive flush'
    print(path.relative_to(DRIVE).as_posix(), digest)
print((SEARCH_OUT / 'selection_report.json').read_text(encoding='utf-8'))
assert_clean()